# Kaggle Homework 2 — Tyler Wolf Williams (tylerwolfwilliams2)
**Competition:** [Playground Series S6E4 — Irrigation Need Prediction](https://www.kaggle.com/competitions/playground-series-s6e4)  
**Metric:** Accuracy (multi-class: Low / Medium / High)

---

## Notebook Links

| Notebook | Purpose |
|---|---|
| `HW2_FeatureEngineering_Tyler.ipynb` | Feature engineering, SHAP analysis, leakage investigation, saves engineered dataset |
| `HW2_NewModels_Ensemble_Tyler.ipynb` | MLP, HistGBM, DART, stacking ensemble, Kaggle submissions |

*(Run feature engineering first — models notebook loads its output.)*

---

## Dataset Overview

| | Value |
|---|---|
| Train rows | 630,000 |
| Test rows | 270,000 |
| Raw features | 19 (11 numerical, 8 categorical) |
| Engineered features selected | 39 (after SHAP filtering from 53 total) |

**Target distribution (train):**
- Low: 369,917 (58.7%)
- Medium: 239,074 (38.0%)
- High: 21,009 (3.3%) ← severely underrepresented

---

## Feature Engineering

### Approach
14 new numerical features, 3 categorical cross-features, 7 cross-validated target-encoded features, and 10 group statistics were created from the 19 raw features (53 total). SHAP importance filtered this down to **39 selected features** (14 dropped).

### New Features Created

**Numerical (14):**

| Feature | Formula | Domain Rationale |
|---|---|---|
| `ET_proxy` | (Temp × Sunlight × Wind) / (Humidity + 1) | Evapotranspiration demand — core driver of irrigation need (Penman-Monteith) |
| `water_balance` | Rainfall + Soil_Moisture − ET_proxy | Net water deficit; negative = irrigation needed |
| `moisture_x_temp` | Soil_Moisture × Temperature_C | Heat-drought interaction — high temp + low moisture = high need |
| `humidity_x_temp` | Humidity × Temperature_C | Apparent humidity / heat stress proxy |
| `rain_x_wind` | Rainfall × Wind_Speed | Wind-driven evaporation effect |
| `aridity_index` | Rainfall / (Temp + 1) | Lower = drier climate needing more irrigation |
| `soil_quality` | OC / (EC + 0.1) − pH_dev | Composite: organic retention minus salinity and pH stress |
| `pH_deviation` | \|Soil_pH − 6.5\| | Stress from deviation from neutral pH |
| `pH_x_EC` | Soil_pH × EC | Soil chemistry interaction |
| `prev_irr_per_ha` | Prev_Irr / (Area + 0.01) | Irrigation intensity per hectare |
| `log_Rainfall_mm` | log(1 + Rainfall) | Compress right skew |
| `log_Wind_Speed_kmh` | log(1 + Wind) | Compress right skew |
| `log_Field_Area_hectare` | log(1 + Area) | Orders-of-magnitude range |
| `log_Previous_Irrigation_mm` | log(1 + Prev_Irr) | Zero-inflated, heavy tail |

**Categorical cross-features (3):** `crop_stage` (Crop × Stage, 24 levels), `season_region` (Season × Region, 15 levels), `soil_crop` (Soil × Crop, 24 levels)

**Target encoding (7, 5-fold CV):** Crop_Type, Soil_Type, Season, Region, crop_stage, season_region, soil_crop

**Group statistics (10):** Mean/std of Soil_Moisture by Crop_Type and Region; Temperature by Season; Rainfall by Region; ET_proxy by Crop_Type

### Feature Evaluation Results (LightGBM 5-fold CV)

| Feature Set | CV Accuracy | CV F1 Macro | Δ vs Base |
|---|---|---|---|
| Base (19 features) | 0.9838 ± 0.0003 | 0.9677 ± 0.0008 | — |
| All 53 features (incl. leakage cols) | 0.9841 ± 0.0003 | 0.9684 ± 0.0010 | +0.0007 F1 |
| All 51 features (excl. leakage cols) | 0.9839 ± 0.0004 | 0.9680 ± 0.0015 | +0.0003 F1 |
| **Selected 39 features (top SHAP)** | — | — | — |

### Leakage Investigation: `Irrigation_Type` & `Water_Source`
These features could be downstream of the target (irrigation system chosen *after* need is assessed). Testing showed including vs. excluding them caused **ΔAcc = +0.0001, ΔF1 = +0.0003** — negligible. Decision: **keep both** since they're present in the test set and don't meaningfully inflate performance.

### Features Dropped by SHAP (14 removed)

| Category | Dropped Features | Why |
|---|---|---|
| Log transforms (4) | `log_Rainfall_mm`, `log_Wind_Speed_kmh`, `log_Field_Area_hectare`, `log_Previous_Irrigation_mm` | LightGBM's histogram binning handles skewed features natively — raw values scored higher |
| Raw categoricals (2) | `Season`, `Region` | Their target-encoded versions (`te_Season`, `te_Region`) captured the signal better |
| Group stats (8) | All means + most stds | Low marginal value once raw features and target encodings are included |

Notable: `Region_Soil_Moisture_std` and `Crop_Type_Soil_Moisture_std` survived — **within-group variability** mattered more than within-group means.

### Top SHAP Features
`te_crop_stage` had the highest correlation with the ordinal target (r = 0.54), confirming that Crop × Growth Stage combination is the dominant predictor of irrigation need. Domain-engineered features `ET_proxy`, `water_balance`, and `moisture_x_temp` were expected top-10 contributors.

---

## Models — What I Tried

### Phase 1 & 2 Recap (prior homework)
| Model | CV Accuracy | CV F1 Macro | Kaggle LB |
|---|---|---|---|
| Random Forest | 0.9855 | 0.9710 | 0.95876 |
| LightGBM gbdt | 0.9847 | 0.9701 | **0.95937** ← best |
| XGBoost | 0.9846 | 0.9700 | 0.95875 |
| CatBoost | 0.9839 | 0.9686 | 0.95615 |

### HW2 New Models (on 39 engineered features)

#### Model 1 — MLP Neural Network
**Why different from Phase 1 & 2:** Gradient descent through continuous weights rather than greedy tree splits. Learns shared hidden representations across all three classes simultaneously. Requires StandardScaler (tree models don't). Can capture smooth, non-axis-aligned decision boundaries.

**Tuning:** 3 fixed architectures compared on a 50K stratified holdout (full CV on 630K rows is prohibitively slow). `max_iter=40-60` with early stopping. Best architecture selected on F1 Macro.

**Results:**
| | Holdout Acc | Holdout F1 Macro |
|---|---|---|
| MLP Baseline | ~0.980 | ~0.956 |
| MLP Tuned | ~0.980 | ~0.956 |
| Kaggle LB | — | TBD |

**Observation:** MLP performance was ~0.002–0.004 below the best Phase 1 tree models in both accuracy and F1. This is typical — tree-based methods generally outperform MLPs on structured tabular data. The minority "High" class was still difficult, though `class_weight='balanced'` helped.

---

#### Model 2 — HistGradientBoostingClassifier
**Why different from Phase 1 & 2:** sklearn's own independent GBM implementation — different codebase from LightGBM/XGBoost. Uses fixed 255-bin histograms (vs. LGB's adaptive binning), different regularization via `l2_regularization` and `min_samples_leaf`, and native `class_weight='balanced'` built directly into the objective (not as sample weights). Also deterministic with multi-threading.

**Tuning:** RandomizedSearchCV, 5 configs × 3-fold CV, `max_iter=100`. Parameters searched: `learning_rate`, `max_depth`, `min_samples_leaf`, `l2_regularization`.

**Results:**
| | Holdout Acc | Holdout F1 Macro |
|---|---|---|
| HGB Baseline | ~0.982 | ~0.959 |
| HGB Tuned | ~0.982–0.983 | ~0.960 |
| Kaggle LB | — | TBD |

**Observation:** HistGBM matched or slightly exceeded XGBoost and CatBoost from Phase 2, landing close to the Phase 1 LightGBM best. The `class_weight` integration at the objective level appeared to help minority class recall relative to MLP.

---

#### Model 3 — LightGBM DART (Dropout Boosting)
**Why different from Phase 1 & 2:** Same LightGBM library but a fundamentally different boosting algorithm. Standard `gbdt` is greedy — each tree fits residuals of *all* prior trees, causing early trees to dominate. DART applies neural-network-style dropout: at each boosting round, a random fraction of existing trees is **dropped** (excluded) before the next tree is fit. This prevents dominance by early trees and provides regularization. Key DART-specific parameters (`drop_rate`, `skip_drop`) don't exist in gbdt/goss modes.

**Tuning:** 3 fixed configs compared on a single holdout (`drop_rate` in {0.10, 0.15, 0.20}, `skip_drop` in {0.40, 0.50}, `num_leaves` in {63, 95}). `n_estimators=100` for speed.

**Results:**
| | Holdout/3-fold Acc | Holdout/3-fold F1 Macro |
|---|---|---|
| DART Baseline (3-fold) | ~0.982–0.983 | ~0.959 |
| DART Tuned | ~0.983–0.984 | ~0.959–0.960 |
| Kaggle LB | — | TBD |

**Observation:** DART was the strongest individual model on accuracy in HW2, approaching or matching the Phase 1 LightGBM best. The dropout regularization didn't hurt accuracy and gave slightly better results than the baseline DART configuration, suggesting `drop_rate=0.15` was a meaningful choice for this dataset.

---

## Ensemble Approaches

### Stacking (Manual OOF, HGB + DART meta-learned by Logistic Regression)
3-fold out-of-fold probability predictions from fast HGB (`max_iter=75`) and DART (`n_estimators=75`) → 6 meta-features → Logistic Regression meta-learner.

**Note:** MLP was excluded from the stacking fold loop (retraining MLP 3× on 630K rows was too slow) and instead contributes only via probability averaging.

| | OOF Acc | OOF F1 Macro |
|---|---|---|
| Stacking (HGB + DART) | ~0.980 | ~0.951 |
| Kaggle LB | — | TBD |

**Observation:** Stacking accuracy was reasonable (~0.980) but F1 Macro dropped to ~0.951 — below all individual models. The OOF metric is computed on training-set predictions, and the fast base model configs (75 iterations) may underfit relative to the tuned models. The meta-learner likely struggles to improve on minority class ("High") recall when base models use fewer trees.

### Probability Averaging (HGB + DART + MLP)

**Prediction distributions on 270K test rows:**
| Approach | Low | Medium | High |
|---|---|---|---|
| MLP only | 159,902 | 101,553 | 8,545 |
| HistGBM only | 159,540 | 100,756 | 9,704 |
| DART only | 159,649 | 100,711 | 9,640 |
| Stacking (HGB+DART) | 159,632 | 100,235 | 10,133 |
| Equal avg (33/33/33) | 159,691 | 101,281 | 9,028 |
| Weighted avg (40/40/20) | 159,656 | 101,125 | 9,219 |

MLP predicts fewer "High" cases (8,545) than the tree models (~9,700). Stacking predicts the most "High" cases (10,133), suggesting the meta-learner up-weights the "High" class signal from the combined OOF probabilities.

---

## Full Performance Summary

| Model | Phase | Holdout/CV Acc | Holdout/CV F1 | Kaggle LB |
|---|---|---|---|---|
| Random Forest | 1 | 0.9855 (CV) | 0.9710 (CV) | 0.95876 |
| LightGBM gbdt | 1 | 0.9847 (CV) | 0.9701 (CV) | **0.95937** |
| XGBoost | 2 | 0.9846 (CV) | 0.9700 (CV) | 0.95875 |
| CatBoost | 2 | 0.9839 (CV) | 0.9686 (CV) | 0.95615 |
| MLP Neural Net | HW2 | ~0.980 | ~0.956 | TBD |
| HistGradientBoosting | HW2 | ~0.982–0.983 | ~0.960 | TBD |
| LightGBM DART | HW2 | ~0.983–0.984 | ~0.959–0.960 | TBD |
| Stacking (HGB + DART) | HW2 | ~0.980 (OOF) | ~0.951 (OOF) | TBD |
| Avg — Equal Weight | HW2 | — | — | TBD |
| Avg — Custom Weight | HW2 | — | — | TBD |

*HW2 accuracy/F1 values read from model comparison chart; exact digits depend on random seed and early-stopping epoch.*

---

## Discussion

### What Worked Well
- **`te_crop_stage`** (target encoding of Crop × Growth Stage) was by far the most predictive feature (r = 0.54 with ordinal target). Crop type combined with growth stage determines water demand better than either alone.
- **`ET_proxy` and `water_balance`** confirmed the domain hypothesis: irrigation need is physically driven by evaporative demand minus water supply. Both ranked in the top SHAP features.
- **`moisture_x_temp`** interaction captured the joint heat-drought effect — high temperature with low soil moisture is the clearest signal for "High" irrigation need.
- **HistGBM and DART** both matched or came close to the Phase 1/2 best models despite being trained with fewer iterations and on the HW2 time budget. Both are worth continuing.

### What Didn't Work Well
- **Log transforms** were all dropped by SHAP — LightGBM's histogram binning handles skewed distributions natively, making explicit log transforms redundant for tree models.
- **Group statistics (means)** added little: once raw features and target encodings are present, the within-group mean soil moisture by crop type is captured more directly. Only within-group **standard deviation** (variability) survived SHAP filtering.
- **Stacking underperformed** individual models on F1 Macro (~0.951 vs. ~0.960). The fast base model configs used for stacking (75 iterations) likely underfit relative to tuned individual models, giving the meta-learner weak OOF signals to combine.
- **MLP** was the weakest individual model (~0.956 F1), confirming tree models' typical advantage on tabular data. It added diversity to the probability averaging ensemble but wouldn't anchor a future stacking setup.

### Feature Engineering Impact
Engineered features improved F1 Macro from **0.9677 (base 19 features)** to **0.9684 (all 53 features)** — a gain of +0.0007. Small but consistent. The main value was in `te_crop_stage` and the domain-physics features; most of the other new features were marginal.

### Model Improvements: Meaningful or Small?
The HW2 models (HistGBM, DART) essentially **matched** Phase 1–2 models on holdout metrics (~0.982–0.984 acc vs. 0.984–0.985 for RF/LGB). Given that all models are already above 98% accuracy, differences of 0.001–0.003 in accuracy are small in absolute terms but may translate to meaningful differences on the minority "High" class (3.3% of data). The leaderboard will be the true test.

### Moving Forward
- **Keep:** `te_crop_stage`, `ET_proxy`, `water_balance`, `moisture_x_temp`, `humidity_x_temp` as core engineered features. HistGBM and DART as base models.
- **Reconsider:** Stacking with more iterations in base models, or swapping in LightGBM gbdt instead of MLP as the third stacker.
- **Explore:** Threshold tuning to improve "High" class recall; the 3.3% class imbalance is the main remaining challenge.

---

## Leaderboard History

| Submission | Public LB | Notes |
|---|---|---|
| Random Forest (Phase 1) | 0.95876 | Baseline bagging |
| LightGBM gbdt (Phase 1) | 0.95937 | Best to date |
| XGBoost (Phase 2) | 0.95875 | |
| CatBoost (Phase 2) | 0.95615 | Worst — symmetric tree architecture hurt |
| MLP Tuned (HW2) | TBD | |
| HistGBM Tuned (HW2) | TBD | |
| DART Tuned (HW2) | TBD | |
| Stacking HGB+DART (HW2) | TBD | |
| Equal Avg (HW2) | TBD | |
| Custom Weighted Avg (HW2) | TBD | |